# deepVOL (CNN-LSTM) 全流程实现

该 Notebook 覆盖从原始 LOB 到 deepVOL 训练的完整流程，并**支持两种数据组织方式**：

1. 普通文件：`data/raw/orderbooks/*.csv|*.gz|*.parquet`
2. 按天压缩包：`data_YYYY-MM-DD.tar.gz`，内部路径支持：
   `/orderbooks/{swap|spot}/{exchange}/{symbol}/{date}.csv`

例如：
- `/orderbooks/swap/binance/BTC_USDT:USDT/2025-12-24.csv`
- `/orderbooks/spot/okx/BTC_USDT/2025-12-24.csv`

因此可以直接读取你当前的 tar.gz 结构。

并兼容如下 LOB 字段命名：`local_ts, exchange_ts, datetime, symbol, bid_p_1..bid_p_10, bid_q_1..bid_q_10, ask_p_1..ask_p_10, ask_q_1..ask_q_10`。


In [ ]:
from __future__ import annotations

import io
import re
import math
import glob
import tarfile
from dataclasses import dataclass
from pathlib import Path
from typing import List, Tuple, Optional

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import classification_report, confusion_matrix


In [ ]:
# ========== 配置 ==========
@dataclass
class Config:
    # 目录模式（普通文件）
    orderbook_dir: Path = Path('data/raw/orderbooks')

    # 压缩包模式（按天 tar.gz）
    archive_dir: Path = Path('.')
    archive_glob: str = 'data_*.tar.gz'

    # 过滤（可选）
    market_type: Optional[str] = None   # 'swap' / 'spot'
    exchange: Optional[str] = None      # 例如 'binance' / 'okx'
    symbol: Optional[str] = None        # 例如 'BTC_USDT:USDT' / 'BTC_USDT' / 'BTC_USD'
    date: Optional[str] = None          # 例如 '2025-12-24'

    # 盘口与 deepVOL 参数
    levels: int = 10
    W: int = 10
    tick_size: float = 0.01

    h: int = 20
    k: int = 5
    T: int = 100

    train_stride: int = 10
    eval_stride: int = 1

    batch_size: int = 64
    lr: float = 1e-3
    epochs: int = 10
    train_ratio: float = 0.7
    val_ratio: float = 0.15
    seed: int = 42


cfg = Config()
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
print(cfg)


In [ ]:
# ========== 列名处理 ==========
def make_orderbook_columns(levels: int = 10) -> List[str]:
    cols = ['timestamp']
    for i in range(1, levels + 1):
        cols += [f'bidprice{i}', f'bidvolume{i}']
    for i in range(1, levels + 1):
        cols += [f'askprice{i}', f'askvolume{i}']
    return cols


def raw_lob_columns_v2(levels: int = 10) -> List[str]:
    cols = ['local_ts', 'exchange_ts', 'datetime', 'symbol']
    for i in range(1, levels + 1):
        cols += [f'bid_p_{i}', f'bid_q_{i}']
    for i in range(1, levels + 1):
        cols += [f'ask_p_{i}', f'ask_q_{i}']
    return cols


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # 若无 header，按常见两种格式补列名
    if all(str(c).isdigit() for c in out.columns):
        if out.shape[1] == len(raw_lob_columns_v2(10)):
            out.columns = raw_lob_columns_v2(10)
        elif out.shape[1] == len(make_orderbook_columns(10)):
            out.columns = make_orderbook_columns(10)

    out.columns = [str(c).strip().lower() for c in out.columns]

    # 将新字段命名映射到 canonical 命名
    rename = {}
    for i in range(1, 11):
        rename[f'bid_p_{i}'] = f'bidprice{i}'
        rename[f'bid_q_{i}'] = f'bidvolume{i}'
        rename[f'ask_p_{i}'] = f'askprice{i}'
        rename[f'ask_q_{i}'] = f'askvolume{i}'
    out = out.rename(columns=rename)

    # 统一 timestamp
    if 'timestamp' not in out.columns:
        if 'local_ts' in out.columns:
            out['timestamp'] = out['local_ts']
        elif 'exchange_ts' in out.columns:
            out['timestamp'] = out['exchange_ts']
        elif 'datetime' in out.columns:
            dt = pd.to_datetime(out['datetime'], errors='coerce', utc=True)
            out['timestamp'] = dt.view('int64')

    return out


In [ ]:
# ========== 数据源发现：支持普通文件 + tar.gz 内部成员 ==========
Source = Tuple[str, Path, Optional[str]]
# kind='file'  -> (kind, file_path, None)
# kind='tar'   -> (kind, tar_path, member_name)


def _norm_symbol(s: str) -> str:
    # 统一比较格式：忽略大小写、下划线、中划线、冒号
    return re.sub(r'[:_\-]', '', s).upper()


def _match_member(member_name: str, cfg: Config) -> bool:
    # 支持路径：/orderbooks/{swap|spot}/{exchange}/{symbol}/{date}.csv
    name = member_name.lstrip('/')
    if '/orderbooks/' not in '/' + name:
        return False
    if not name.endswith('.csv'):
        return False

    parts = name.split('/')
    # 容错：路径可能有前缀目录，找到 orderbooks 下标
    try:
        i = parts.index('orderbooks')
        market_type = parts[i + 1]   # swap / spot
        exchange = parts[i + 2]
        symbol = parts[i + 3]
        date_file = parts[i + 4]
    except Exception:
        return False

    date_only = date_file.replace('.csv', '')

    if cfg.market_type and market_type.lower() != cfg.market_type.lower():
        return False
    if cfg.exchange and exchange.lower() != cfg.exchange.lower():
        return False
    if cfg.symbol and _norm_symbol(symbol) != _norm_symbol(cfg.symbol):
        return False
    if cfg.date and date_only != cfg.date:
        return False
    return True


def list_sources(cfg: Config) -> List[Source]:
    sources: List[Source] = []

    # A) 普通文件
    for ptn in ['*.parquet', '*.csv', '*.gz', '*.csv.gz']:
        for f in glob.glob(str(cfg.orderbook_dir / ptn)):
            sources.append(('file', Path(f), None))

    # B) tar.gz 压缩包
    archives = sorted(Path(x) for x in glob.glob(str(cfg.archive_dir / cfg.archive_glob)))
    for tar_path in archives:
        try:
            with tarfile.open(tar_path, 'r:gz') as tf:
                for m in tf.getmembers():
                    if not m.isfile():
                        continue
                    if _match_member(m.name, cfg):
                        sources.append(('tar', tar_path, m.name))
        except tarfile.TarError as e:
            print(f'[WARN] 跳过损坏压缩包 {tar_path}: {e}')

    uniq = sorted(set(sources), key=lambda x: (x[0], str(x[1]), x[2] or ''))
    return uniq


sources = list_sources(cfg)
print(f'发现 {len(sources)} 个可用数据源')
for s in sources[:10]:
    print(s)


In [ ]:
# ========== 数据读取 ==========
def load_source(source: Source) -> pd.DataFrame:
    kind, p, member = source

    if kind == 'file':
        suffix = ''.join(p.suffixes).lower()
        if suffix.endswith('.parquet'):
            df = pd.read_parquet(p)
        elif suffix.endswith('.csv') or suffix.endswith('.csv.gz') or suffix.endswith('.gz'):
            try:
                df = pd.read_csv(p, sep=None, engine='python')
            except Exception:
                df = pd.read_csv(p, header=None)
        else:
            raise ValueError(f'Unsupported file: {p}')
        return normalize_columns(df)

    if kind == 'tar':
        assert member is not None
        with tarfile.open(p, 'r:gz') as tf:
            fobj = tf.extractfile(member)
            if fobj is None:
                raise FileNotFoundError(f'Cannot extract {member} from {p}')
            raw = fobj.read()
            bio = io.BytesIO(raw)
            try:
                df = pd.read_csv(bio, sep=None, engine='python')
            except Exception:
                bio.seek(0)
                df = pd.read_csv(bio, header=None)
        return normalize_columns(df)

    raise ValueError(f'Unknown source type: {kind}')


In [ ]:
# ========== 清洗 ==========
def clean_orderbook(df: pd.DataFrame) -> pd.DataFrame:
    required = ['timestamp', 'bidprice1', 'askprice1']
    for c in required:
        if c not in df.columns:
            raise KeyError(f'Missing required column: {c}')

    out = df.copy()
    for c in out.columns:
        out[c] = pd.to_numeric(out[c], errors='coerce')

    out = out.dropna(subset=['timestamp', 'bidprice1', 'askprice1'])
    out = out[out['askprice1'] > out['bidprice1']]
    out = out.sort_values('timestamp').groupby('timestamp', as_index=False).last()

    # 掐头去尾（按时长比例近似去掉首尾各10分钟）
    t0, t1 = out['timestamp'].min(), out['timestamp'].max()
    if t1 > t0:
        left = t0 + (t1 - t0) * (10 / (24 * 60))
        right = t1 - (t1 - t0) * (10 / (24 * 60))
        out = out[(out['timestamp'] >= left) & (out['timestamp'] <= right)]

    return out.reset_index(drop=True)


In [ ]:
# ========== deepVOL 特征 ==========
def build_volume_features(clean_df: pd.DataFrame, W: int, tick_size: float, levels: int = 10) -> np.ndarray:
    N = len(clean_df)
    feat = np.zeros((N, 2 * W), dtype=np.float32)

    bps = [f'bidprice{i}' for i in range(1, levels + 1)]
    bvs = [f'bidvolume{i}' for i in range(1, levels + 1)]
    aps = [f'askprice{i}' for i in range(1, levels + 1)]
    avs = [f'askvolume{i}' for i in range(1, levels + 1)]

    for t in range(N):
        row = clean_df.iloc[t]
        mid = (row['askprice1'] + row['bidprice1']) / 2.0

        for p_col, v_col in zip(bps, bvs):
            p, v = row[p_col], row[v_col]
            if pd.isna(p) or pd.isna(v):
                continue
            d = int(math.floor((mid - p) / tick_size))
            if 1 <= d <= W:
                feat[t, W - d] += float(v)

        for p_col, v_col in zip(aps, avs):
            p, v = row[p_col], row[v_col]
            if pd.isna(p) or pd.isna(v):
                continue
            d = int(math.floor((p - mid) / tick_size))
            if 1 <= d <= W:
                feat[t, W + d - 1] += float(v)

    return feat


In [ ]:
# ========== 标签 ==========
def compute_returns(mid: pd.Series, h: int, k: int) -> pd.Series:
    future = mid.rolling(window=2*k+1, center=True, min_periods=2*k+1).mean().shift(-h)
    return (future - mid) / mid


def compute_gamma(train_returns: np.ndarray) -> float:
    train_returns = train_returns[~np.isnan(train_returns)]
    q33, q66 = np.quantile(train_returns, [0.33, 0.66])
    return float((abs(q33) + q66) / 2.0)


def discretize_labels(returns: np.ndarray, gamma: float) -> np.ndarray:
    y = np.full_like(returns, fill_value=-1, dtype=np.int64)
    valid = ~np.isnan(returns)
    y[(returns < -gamma) & valid] = 0
    y[(np.abs(returns) <= gamma) & valid] = 1
    y[(returns > gamma) & valid] = 2
    return y


In [ ]:
# ========== 滑窗构建 ==========
def create_dataset(volume_features: np.ndarray, labels: np.ndarray, T: int, W: int, stride: int = 1):
    X_list, y_list = [], []
    for t in range(T, len(volume_features), stride):
        if labels[t] < 0:
            continue
        window = volume_features[t-T:t]
        local_max = np.max(window)
        if local_max <= 0:
            continue

        window = window / local_max
        bid, ask = window[:, :W], window[:, W:]
        x = np.stack([bid, ask], axis=-1).astype(np.float32)   # (T, W, 2)

        X_list.append(x)
        y_list.append(labels[t])

    if not X_list:
        return np.empty((0, T, W, 2), dtype=np.float32), np.empty((0,), dtype=np.int64)
    return np.stack(X_list, axis=0), np.asarray(y_list, dtype=np.int64)


In [ ]:
# ========== 单源样本准备 ==========
def prepare_one_source(source: Source, cfg: Config):
    raw = load_source(source)
    clean = clean_orderbook(raw)
    if len(clean) < cfg.T + cfg.h + 2*cfg.k + 5:
        return None

    feat = build_volume_features(clean, W=cfg.W, tick_size=cfg.tick_size, levels=cfg.levels)
    mid = (clean['askprice1'] + clean['bidprice1']) / 2.0
    rets = compute_returns(mid, h=cfg.h, k=cfg.k).values

    N = len(clean)
    tr_end = int(N * cfg.train_ratio)
    va_end = int(N * (cfg.train_ratio + cfg.val_ratio))

    gamma = compute_gamma(rets[:tr_end])
    labels = discretize_labels(rets, gamma)

    Xtr, ytr = create_dataset(feat[:tr_end], labels[:tr_end], cfg.T, cfg.W, cfg.train_stride)
    Xva, yva = create_dataset(feat[tr_end:va_end], labels[tr_end:va_end], cfg.T, cfg.W, cfg.eval_stride)
    Xte, yte = create_dataset(feat[va_end:], labels[va_end:], cfg.T, cfg.W, cfg.eval_stride)

    info = {
        'source': f'{source[0]}::{source[1]}::{source[2]}',
        'rows_raw': len(raw),
        'rows_clean': len(clean),
        'gamma': gamma,
        'train_samples': len(ytr),
        'val_samples': len(yva),
        'test_samples': len(yte),
    }
    return Xtr, ytr, Xva, yva, Xte, yte, info


In [ ]:
# ========== 多源聚合 ==========
parts, infos = [], []
for s in sources:
    out = prepare_one_source(s, cfg)
    if out is None:
        continue
    Xtr, ytr, Xva, yva, Xte, yte, info = out
    if len(ytr) == 0:
        continue
    parts.append((Xtr, ytr, Xva, yva, Xte, yte))
    infos.append(info)

if not parts:
    print('未生成样本，请检查：archive_dir/archive_glob/exchange/symbol/date/tick_size。')
else:
    X_train = np.concatenate([p[0] for p in parts], axis=0)
    y_train = np.concatenate([p[1] for p in parts], axis=0)
    X_val = np.concatenate([p[2] for p in parts], axis=0)
    y_val = np.concatenate([p[3] for p in parts], axis=0)
    X_test = np.concatenate([p[4] for p in parts], axis=0)
    y_test = np.concatenate([p[5] for p in parts], axis=0)

    print('X_train:', X_train.shape, 'y_train:', y_train.shape)
    print('X_val  :', X_val.shape, 'y_val  :', y_val.shape)
    print('X_test :', X_test.shape, 'y_test :', y_test.shape)
    display(pd.DataFrame(infos).head())


In [ ]:
# ========== 模型 ==========
class DeepVOLDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.from_numpy(X).float().unsqueeze(1)  # (N,1,T,W,2)
        self.y = torch.from_numpy(y).long()
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class InceptionTime(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        b = out_ch // 4
        self.b1 = nn.Conv2d(in_ch, b, kernel_size=(1,1), padding=(0,0))
        self.b3 = nn.Conv2d(in_ch, b, kernel_size=(3,1), padding=(1,0))
        self.b5 = nn.Conv2d(in_ch, b, kernel_size=(5,1), padding=(2,0))
        self.bp = nn.Sequential(
            nn.MaxPool2d(kernel_size=(3,1), stride=1, padding=(1,0)),
            nn.Conv2d(in_ch, out_ch - 3*b, kernel_size=(1,1), padding=(0,0)),
        )
        self.act = nn.ReLU(inplace=True)
    def forward(self, x):
        return self.act(torch.cat([self.b1(x), self.b3(x), self.b5(x), self.bp(x)], dim=1))


class DeepVOLNet(nn.Module):
    def __init__(self, n_classes: int = 3, lstm_hidden: int = 64):
        super().__init__()
        self.conv3d = nn.Sequential(
            nn.Conv3d(1, 32, kernel_size=(1,1,2), stride=(1,1,1)),
            nn.ReLU(inplace=True),
        )
        self.spatial = nn.Sequential(
            nn.Conv2d(32, 32, kernel_size=(1,2), padding=(0,0)), nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=(3,1), padding=(1,0)), nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=(1,2), padding=(0,0)), nn.ReLU(inplace=True),
        )
        self.inception = InceptionTime(32, 64)
        self.lstm = nn.LSTM(input_size=64, hidden_size=lstm_hidden, batch_first=True)
        self.fc = nn.Linear(lstm_hidden, n_classes)

    def forward(self, x):
        x = self.conv3d(x).squeeze(-1)
        x = self.inception(self.spatial(x))
        x = x.mean(dim=-1).transpose(1, 2)   # (B,T,64)
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])


In [ ]:
# ========== 训练 ==========
def run_epoch(model, loader, criterion, optimizer=None, device='cpu'):
    train = optimizer is not None
    model.train() if train else model.eval()
    loss_sum, n, correct = 0.0, 0, 0

    for X, y in loader:
        X, y = X.to(device), y.to(device)
        with torch.set_grad_enabled(train):
            logits = model(X)
            loss = criterion(logits, y)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        bs = y.size(0)
        loss_sum += loss.item() * bs
        n += bs
        correct += (logits.argmax(1) == y).sum().item()

    return loss_sum / max(n, 1), correct / max(n, 1)


def predict(model, loader, device='cpu'):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for X, y in loader:
            logits = model(X.to(device))
            ys.append(y.numpy())
            ps.append(logits.argmax(1).cpu().numpy())
    return np.concatenate(ys), np.concatenate(ps)


device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device =', device)

if 'X_train' in globals() and len(y_train) > 0:
    train_loader = DataLoader(DeepVOLDataset(X_train, y_train), batch_size=cfg.batch_size, shuffle=True)
    val_loader = DataLoader(DeepVOLDataset(X_val, y_val), batch_size=cfg.batch_size, shuffle=False)
    test_loader = DataLoader(DeepVOLDataset(X_test, y_test), batch_size=cfg.batch_size, shuffle=False)

    model = DeepVOLNet().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)

    best_val, best_state = float('inf'), None
    for ep in range(1, cfg.epochs + 1):
        tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer, device)
        va_loss, va_acc = run_epoch(model, val_loader, criterion, None, device)
        print(f'Epoch {ep:02d} | train {tr_loss:.4f}/{tr_acc:.4f} | val {va_loss:.4f}/{va_acc:.4f}')
        if va_loss < best_val:
            best_val = va_loss
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    y_true, y_pred = predict(model, test_loader, device)
    print('\n[TEST] Confusion Matrix')
    print(confusion_matrix(y_true, y_pred))
    print('\n[TEST] Classification Report')
    print(classification_report(y_true, y_pred, digits=4))

    Path('models').mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), 'models/deepvol_cnn_lstm.pt')
    print('模型已保存到 models/deepvol_cnn_lstm.pt')
else:
    print('当前未构建出训练样本，跳过训练。')
